In [7]:

import requests
import pandas as pd



def fetch_api_data():
    """
    Fetches recipe data directly from the API and processes it for Pinecone.
    """
    api_url = 'https://apis.delicut.ae/api/v1/recipes/fetch-all-weekly'

    try:
        response = requests.get(api_url)
        if response.status_code != 200:
            print(f"Error fetching API: {response.status_code}")
            return []

        data = response.json()
        recipes_by_category = data.get("data", {})

        all_recipes = []
        for category, recipes in recipes_by_category.items():
            if isinstance(recipes, list):
                for recipe in recipes:
                    recipe["meal_category"] = category  # Preserve category

                    # Remove unwanted fields
                    for field in ["category_type", "protein_category_info", "website_image",
                                  "cooking_complexity", "plating_complexity", "highly_perishable"]:
                        recipe.pop(field, None)

                    # Merge ingredients and remove duplicates
                    unique_ingredients = set()
                    for ingredient_group in recipe.get("ingredients", []):
                        unique_ingredients.update(ingredient_group.get("ingredients", []))
                    recipe["ingredients"] = list(unique_ingredients)

                    # Filter variants to retain only required fields
                    filtered_variants = []
                    for variant in recipe.get("variants", []):
                        filtered_variant = {
                            "protein_category": variant.get("protein_category"),
                            "protein_option": variant.get("protein_option"),
                            "size": variant.get("size"),
                            "variant_ingredients": variant.get("variant_ingredients", []),
                            "kcal": variant.get("kcal"),
                            "fat": variant.get("fat"),
                            "carb": variant.get("carb"),
                            "protein": variant.get("protein"),
                            "allergens": variant.get("allergens", [])
                        }
                        filtered_variants.append(filtered_variant)
                    recipe["variants"] = filtered_variants

                    all_recipes.append(recipe)

        return all_recipes

    except requests.exceptions.RequestException as e:
        print(f"Error fetching API: {str(e)}")
        return []


In [8]:
recipes = fetch_api_data()  # Fetch from API


In [9]:
recipes

[{'recipe_id': '668f9c593d6d34934a270d36',
  'meal_category': 'meal',
  'dish_name': 'Asian Rice Noodles With Stir-fry',
  'cuisine': 'Asian',
  'dish_type': ['Noodles'],
  'description': 'Coconut-rich sauce in stirfry style with rice noodles & vegetables. Dairy & Gluten Free.',
  'ingredients': ['White Pepper Powder',
   'Onion',
   'Dried Red Chilli',
   'Lemon Grass',
   'Sunflower Oil',
   'Coconut Oil',
   'Coconut Milk Powder',
   'Bell Pepper',
   'Olive Oil',
   'Pumpkin',
   'Water',
   'Lemon Juice',
   'Red Chilli',
   'Coriander',
   'Coriander Seeds',
   'Galangal',
   'Garlic',
   'Sticky Rice Noodles',
   'Cumin Seeds',
   'Zucchini',
   'Carrot',
   'Salt'],
  'variants': [{'protein_category': 'balance',
    'protein_option': 'Tofu (Veg)',
    'size': 'small',
    'variant_ingredients': ['Coconut Milk Powder',
     'Coriander',
     'Coriander Seeds',
     'Cumin Seeds',
     'Galangal',
     'Garlic',
     'Lemon Grass',
     'Lemon Juice',
     'Onion',
     'Onion',


# **Pinecone Push**

In [ ]:

import os
import json
from dotenv import load_dotenv
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from recipe_processing import fetch_api_data

# Load environment variables
load_dotenv(override=True)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-index"
local_file_path = "yytest/recipes.json"  # Local storage file

def initialize_pinecone():
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=384,  # Match the embedding model output size
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
    return pc.Index(index_name)

index = initialize_pinecone()

# Load embedding model
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def flatten_allergens(recipe):
    """
    Flattens allergens from all variants into a single list of unique allergens.
    """
    allergens = set()
    for variant in recipe.get("variants", []):
        allergens.update(variant.get("allergens", []))  # Add allergens from each variant
    
    # Return allergens as a sorted list
    return list(sorted(allergens))

def flatten_fields(recipe, fields):
    """
    Flattens specified fields from all variants into separate lists of unique values.
    
    :param recipe: Dictionary containing recipe data with variants.
    :param fields: List of fields to flatten from the variants.
    :return: Dictionary with each field containing a sorted list of unique values.
    """
    flattened_data = {field: set() for field in fields}

    for variant in recipe.get("variants", []):
        for field in fields:
            flattened_data[field].update(variant.get(field, []))  # Add values from each variant
    
    # Convert sets to sorted lists
    return {field: sorted(values) for field, values in flattened_data.items()}


def format_variants(variants):
    """
    Removes duplicate variants based on nutritional values (kcal, carb, fat, protein)
    and formats the unique variants for storage.
    """
    # Remove duplicates based on the nutritional values (kcal, carb, fat, protein)
    unique_variants = []
    seen = set()

    for variant in variants:
        # Create a tuple of the nutritional values to identify unique variants
        variant_tuple = (variant.get('kcal'), variant.get('carb'), variant.get('fat'), variant.get('protein'))
        
        if variant_tuple not in seen:
            seen.add(variant_tuple)
            unique_variants.append(variant)

    # Format the unique variants into a string (kcal, carb, fat, protein)
    formatted_variants = [
        f"kcal: {v['kcal']}, carb: {v['carb']}, fat: {v['fat']}, protein: {v['protein']}"
        for v in unique_variants
    ]
    
    # Join all formatted variants into a single string for storing in Pinecone
    formatted_variants_str = " | ".join(formatted_variants)
    
    return formatted_variants_str

    
def process_recipe_for_pinecone(recipe):
    """
    Converts a recipe into an embedding-friendly format and metadata.
    """
    # Flatten allergens for the recipe
    allergens_flat = flatten_allergens(recipe, "allergens")
    proteinop_flat = flatten_allergens(recipe, "protein_option")
    size_flat = flatten_allergens(recipe, "size")
    proteincategory_flat = flatten_allergens(recipe, "protein_category")

    # Format the variants as a compact string (including kcal, carb, fat, protein)
    formatted_variants_str = format_variants(recipe.get("variants", []))

    # # Format variants as text
    # variant_texts = [
    #     f"Protein Category: {variant.get('protein_category', '')}, "
    #     f"Kcal: {variant.get('kcal', '')}, "
    #     f"Fat: {variant.get('fat', '')}, "
    #     f"Carb: {variant.get('carb', '')}, "
    #     f"Protein: {variant.get('protein', '')}, "
    #     f"Allergens: {', '.join(variant.get('allergens', []))}."
    #     for variant in recipe.get("variants", [])
    # ]
    # variants_text = " | ".join(variant_texts)

    # Generate a text representation for embedding
    text = f"Dish Name: {recipe.get('dish_name', '')}. " \
           f"Description: {recipe.get('description', '')}. " \
           f"Ingredients: {', '.join(recipe.get('ingredients', []))}. " \
           f"Meal Category: {recipe.get('meal_category', '')}. " \
           f"Cuisine: {recipe.get('cuisine', '')}. " \
           f"Dish Type: {', '.join(recipe.get('dish_type', []))}. " \
           f"Spice Level: {recipe.get('spice_level', '')}. " \
           f"Variants: {formatted_variants_str}"  # Use the formatted variants string here

    # Create embedding
    vector = embed_model.embed_query(text)

    # # Convert `variants` to JSON string
    # variants_json = json.dumps(recipe.get("variants", []), ensure_ascii=False).replace('"', "'")

    # # Convert `variants` to a string with single quotes and preserve the structure
    # variants = recipe.get("variants", [])

    # # Convert to string representation with single quotes (instead of JSON format)
    # variants_str = str(variants).replace('"', "'")

    # # Now the variants_str will have the desired format

    # Prepare metadata
    metadata = {
        "recipe_id": recipe.get("recipe_id", ""),
        "meal_category": recipe.get("meal_category", ""),
        "dish_name": recipe.get("dish_name", ""),
        "cuisine": recipe.get("cuisine", ""),
        "dish_type": recipe.get("dish_type", []),
        "description": recipe.get("description", ""),
        "ingredients": recipe.get("ingredients", []),
        "variants": formatted_variants_str,  # Store the formatted variants as a string
        "spice_level": recipe.get("spice_level", ""),
        "text": text,
        "allergens": allergens_flat  # Add flattened allergens to metadata
    }

    return vector, metadata

def save_to_pinecone(recipes):
    """
    Converts recipes into embeddings and saves them to Pinecone.
    """
    # Save locally first
    # save_to_local(recipes)

    # Clear Pinecone index
    index_stats = index.describe_index_stats()
    if index_stats['total_vector_count'] > 0:
        print("Index is not empty, clearing the index...")
        index.delete(deleteAll=True)
    else:
        print("Index is already empty or does not exist.")

    vectors_to_upsert = [
        (recipe["recipe_id"], *process_recipe_for_pinecone(recipe))
        for recipe in recipes
    ]

    if vectors_to_upsert:
        index.upsert(vectors=vectors_to_upsert)
        print(f"✅ Successfully saved {len(vectors_to_upsert)} recipes to Pinecone.")

# Main execution
if __name__ == "__main__":
    recipes = fetch_api_data()  # Fetch from API
    save_to_pinecone(recipes)  # Save locally & to Pinecone

In [10]:
import os
import json
from dotenv import load_dotenv
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec

# Load environment variables
load_dotenv(override=True)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"
# local_file_path = "yytest/recipes.json"  # Local storage file

def initialize_pinecone():
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=384,  # Match the embedding model output size
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
    return pc.Index(index_name)

index = initialize_pinecone()

# Load HuggingFace Embedding Model
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def generate_embedding(text):
    """
    Generate embeddings using HuggingFace sentence transformer.
    """
    return embed_model.embed_query(text)  # Convert text to embedding vector

def format_and_push_to_pinecone(recipes):
    """
    Processes multiple recipes and pushes each variant as a separate vector to Pinecone.
    
    :param recipes: List of recipe dictionaries.
    """
    vectors = []

    for recipe in recipes:
        for variant in recipe.get("variants", []):
            # Combine main ingredients + variant ingredients (keep unique values)
            all_ingredients = list(set(recipe["ingredients"] + variant.get("variant_ingredients", [])))

            # Construct the text field for embedding
            text = f"Dish Name: {recipe.get('dish_name', '')}. " \
                   f"Description: {recipe.get('description', '')}. " \
                   f"Ingredients: {', '.join(all_ingredients)}. " \
                   f"Meal Category: {recipe.get('meal_category', '')}. " \
                   f"Cuisine: {recipe.get('cuisine', '')}. " \
                   f"Dish Type: {', '.join(recipe.get('dish_type', []))}. " \
                   f"Spice Level: {recipe.get('spice_level', '')}. " \
                   f"Protein Category: {variant.get('protein_category', '')}. " \
                   f"Protein Option: {variant.get('protein_option', '')}. " \
                   f"Size: {variant.get('size', '')}. " \
                   f"Calories: {variant.get('kcal', '')}. " \
                   f"Fat: {variant.get('fat', '')}. " \
                   f"Carbohydrates: {variant.get('carb', '')}. " \
                   f"Protein: {variant.get('protein', '')}. " \
                   f"Allergens: {', '.join(variant.get('allergens', []))}. "

            metadata = {
                "recipe_id": recipe["recipe_id"],
                "meal_category": recipe["meal_category"],
                "dish_name": recipe["dish_name"],
                "cuisine": recipe["cuisine"],
                "dish_type": recipe["dish_type"],
                "description": recipe["description"],
                "ingredients": all_ingredients,  # Unique combined ingredients
                "spice_level": recipe["spice_level"],
                "protein_category": variant["protein_category"],  # Variant-specific
                "protein_option": variant["protein_option"],      # Variant-specific
                "size": variant["size"],                          # Variant-specific
                "kcal": variant["kcal"],                          # Variant-specific
                "fat": variant["fat"],                            # Variant-specific
                "carb": variant["carb"],                          # Variant-specific
                "protein": variant["protein"],                    # Variant-specific
                "allergens": variant["allergens"],                # Variant-specific
                "text": text  # Text field for embedding
            }

            # Generate a unique vector ID (e.g., "669785453d6d34934a276cb8_balance_standard")
            vector_id = f"{recipe['recipe_id']}_{variant['protein_category']}_{variant['protein_option']}"

            # Generate embedding for this variant
            embedding = generate_embedding(text)

            # Prepare vector data
            vector_data = {
                "id": vector_id,
                "values": embedding,  # Embedding vector
                "metadata": metadata
            }

            vectors.append(vector_data)

    # Push all vectors to Pinecone
    index.upsert(vectors)


/home/hello/sb/mealrec/venv/lib/python3.10/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
/tmp/ipykernel_19177/3533569449.py:31: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [11]:
format_and_push_to_pinecone(recipes)


In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec

# Load environment variables
load_dotenv(override=True)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"
# local_file_path = "yytest/recipes.json"  # Local storage file

def initialize_pinecone():
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=384,  # Match the embedding model output size
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
    return pc.Index(index_name)

index = initialize_pinecone()

# Load HuggingFace Embedding Model
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def generate_embedding(text):
    """
    Generate embeddings using HuggingFace sentence transformer.
    """
    return embed_model.embed_query(text)  # Convert text to embedding vector
def format_and_push_to_pinecone_2(recipes):
    """
    Processes multiple recipes and pushes each variant as a separate vector to Pinecone.
    """
    vectors = []
    total_variants = 0  # Count the number of variants processed

    for recipe in recipes:
        recipe_variants = recipe.get("variants", [])  # Get variants
        total_variants += len(recipe_variants)  # Count variants
        
        print(f"🔍 Processing Recipe: {recipe.get('dish_name', 'Unknown')} - Found {len(recipe_variants)} variants")

        for variant in recipe_variants:
            # Combine main ingredients + variant ingredients (keep unique values)
            all_ingredients = list(set(recipe["ingredients"] + variant.get("variant_ingredients", [])))

            # Construct the text field for embedding
            text = f"Dish Name: {recipe.get('dish_name', '')}. " \
                   f"Description: {recipe.get('description', '')}. " \
                   f"Ingredients: {', '.join(all_ingredients)}. " \
                   f"Meal Category: {recipe.get('meal_category', '')}. " \
                   f"Cuisine: {recipe.get('cuisine', '')}. " \
                   f"Dish Type: {', '.join(recipe.get('dish_type', []))}. " \
                   f"Spice Level: {recipe.get('spice_level', '')}. " \
                   f"Protein Category: {variant.get('protein_category', '')}. " \
                   f"Protein Option: {variant.get('protein_option', '')}. " \
                   f"Size: {variant.get('size', '')}. " \
                   f"Calories: {variant.get('kcal', '')}. " \
                   f"Fat: {variant.get('fat', '')}. " \
                   f"Carbohydrates: {variant.get('carb', '')}. " \
                   f"Protein: {variant.get('protein', '')}. " \
                   f"Allergens: {', '.join(variant.get('allergens', []))}. "

            metadata = {
                "recipe_id": recipe["recipe_id"],
                "meal_category": recipe["meal_category"],
                "dish_name": recipe["dish_name"],
                "cuisine": recipe["cuisine"],
                "dish_type": recipe["dish_type"],
                "description": recipe["description"],
                "ingredients": all_ingredients,  # Unique combined ingredients
                "spice_level": recipe["spice_level"],
                "protein_category": variant["protein_category"],  # Variant-specific
                "protein_option": variant["protein_option"],      # Variant-specific
                "size": variant["size"],                          # Variant-specific
                "kcal": variant["kcal"],                          # Variant-specific
                "fat": variant["fat"],                            # Variant-specific
                "carb": variant["carb"],                          # Variant-specific
                "protein": variant["protein"],                    # Variant-specific
                "allergens": variant["allergens"],                # Variant-specific
                "text": text  # Text field for embedding
            }

            # Generate a unique vector ID (e.g., "669785453d6d34934a276cb8_balance_standard")
            vector_id = f"{recipe['recipe_id']}_{variant['protein_category']}_{variant['protein_option']}_{variant['size']}"

            # Generate embedding for this variant
            embedding = generate_embedding(text)

            # Prepare vector data
            vector_data = {
                "id": vector_id,
                "values": embedding,  # Embedding vector
                "metadata": metadata
            }

            vectors.append(vector_data)

    print(f"\n✅ Total Recipes Processed: {len(recipes)}")
    print(f"✅ Total Variants Processed: {total_variants}")  # Should be 23
    print(f"✅ Total Vectors Ready to Upload: {len(vectors)}")  # Should match variants

    # Debugging: Print the metadata for all recipes
    for vector in vectors:
        print(json.dumps(vector['metadata'], indent=2))

    # Clear Pinecone index
    index_stats = index.describe_index_stats()
    if index_stats['total_vector_count'] > 0:
        print("Index is not empty, clearing the index...")
        index.delete(deleteAll=True)
    else:
        print("Index is already empty or does not exist.")

    # Push all vectors to Pinecone
    index.upsert(vectors)




In [26]:
format_and_push_to_pinecone_2(recipes)

🔍 Processing Recipe: Asian Rice Noodles With Stir-fry - Found 14 variants
🔍 Processing Recipe: Butter Chickpeas with Rice - Found 14 variants
🔍 Processing Recipe: Mushroom Penne - Found 14 variants
🔍 Processing Recipe: Cajun Roasted Chicken Sandwich - Found 9 variants
🔍 Processing Recipe: Creamy Tomato Tagliatelle  - Found 20 variants
🔍 Processing Recipe: Greek Salad - Found 14 variants
🔍 Processing Recipe: Beetroot & Quinoa Salad - Found 24 variants
🔍 Processing Recipe: Pumpkin & Yoghurt Bowl - Found 20 variants
🔍 Processing Recipe: Tuscan Protein & Black Rice - Found 24 variants
🔍 Processing Recipe: Lemon Quinoa & Green Beans with Roasted Protein - Found 9 variants
🔍 Processing Recipe: Beef Stew with Saffron Rice - Found 10 variants
🔍 Processing Recipe: Senegalese Protein & Tomato Rice - Found 24 variants
🔍 Processing Recipe: Mushroom Meatballs & Mash - Found 24 variants
🔍 Processing Recipe: Peruvian Chicken & Mash Potato - Found 10 variants
🔍 Processing Recipe: Machboos Rubyan - Fou